[![](imagens/colab-badge.png){width="16%"}](https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/py.pt/cap05/cap05.EPs_aluno.ipynb)
[![](imagens/github-badge.png){width="19%"}](https://github.com/fzampirolli/pdi-vc)

## 💻 **Parte Prática com Exercícios de Programação**

🚧 **Em construção!**

A presente lista de exercícios de programação (EP) consolida as formulações teóricas apresentadas ao longo do Capítulo 5 — Transformadas e Compressão — por meio de uma trilha prática aplicada. Os exercícios são estruturados a partir de matrizes de dimensões reduzidas, viabilizando a validação analítica e a inspeção manual de cada coeficiente, mantendo a consistência metodológica adotada nos capítulos anteriores.

O encadeamento dos exercícios reproduz rigorosamente o fluxo conceitual do capítulo: inicia-se com a implementação explícita da Transformada Discreta de Fourier (DFT) a partir de sua definição matemática fundamental; avança-se para o projeto de filtros passa-baixa e máscaras *notch* no domínio da frequência; aplica-se a quantização de coeficientes (núcleo da compressão com perda); e conclui-se com a integração dessas etapas na construção de um *pipeline* de compressão JPEG simplificado e na análise perceptual de formatos de imagem.

::: {.callout-important}
### Diretrizes para a Resolução dos Exercícios de Programação {.unnumbered}

Em todos os exercícios deste capítulo, as coordenadas do **centro do espectro** (origem das frequências espaciais pós-aplicação do deslocamento `fftshift`) devem ser determinadas via divisão inteira. Para uma matriz com $L$ linhas e $C$ colunas, a componente de frequência nula localiza-se na posição:

$$
(c_y, c_x) = \left( \left\lfloor \frac{L}{2} \right\rfloor, \left\lfloor \frac{C}{2} \right\rfloor \right)
$$

Esta convenção é rigorosamente idêntica à adotada pela função `np.fft.fftshift`. Ademais, em todas as etapas que exijam discretização ou arredondamento numérico (seja na quantização de coeficientes AC ou na reconstrução final de pixels), deve-se empregar o arredondamento padrão para o inteiro mais próximo (*round half away from zero*), mitigando ambiguidades em valores com fração exatamente igual a $0.5$.
:::


### 🎯 Objetivo deste Caderno {.unnumbered}

O caderno permite desenvolver, validar, organizar e testar soluções de **Exercícios de Programação (EPs)** em ambientes interativos, como o Colab, com os mesmos casos de teste do Moodle, copiando para lá apenas na hora de registrar a nota oficial.

#### *Download* {.unnumbered}

Baixe `morph.py` e `testsuite.py` executando a célula abaixo:

In [2]:
#| quarto-raw: true
import os, urllib.request

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

import config
config.setup(testsuite=True)
from morph import mm
from testsuite import TestSuite

✅ Ambiente pronto. Morph: 1.1.8 | OpenCV: 5.0.0 | TestSuite: 1.1.2


#### Executando os Testes {.unnumbered}
Para avaliar os testes, execute `TestSuite("EP05_01.extensão").run()` numa nova célula, trocando a extensão pela da linguagem usada (`.py`, `.java`, `.c`, `.cpp`, `.js` ou `.r`). O sistema baixa os casos de teste do GitHub, executa o programa e calcula a nota automaticamente.

Para testar código Python diretamente, sem salvar arquivo, use `run_code(codigo)` passando o código como *string* numa variável `codigo`:

```python
codigo = """
from morph import mm
# ... seu código aqui ...
"""
TestSuite("EP05_01").run_code(codigo)
```

### EP05_01 🟢 Filtro Passa-Baixa Ideal por Distância no Espectro

Em um ***scanner* de documentos antigo**, o sensor capta papel amassado e textura de fibra junto com o texto — ruído de alta frequência que "polui" o espectro nas bordas. O técnico de manutenção não tem acesso à imagem original, apenas ao **espectro de magnitude já calculado** pelo software do *scanner*. Seu trabalho é simples e cirúrgico: manter apenas o **círculo central** de baixas frequências (a estrutura global do documento) e apagar tudo que estiver fora do raio $D_0$, eliminando a textura fina sem nem precisar tocar na imagem espacial.

Este é o **Filtro Passa-Baixa Ideal (LPFI)**: a operação espectral mais direta do capítulo, mas também a que melhor revela a anatomia de um espectro centrado.

#### 📋 Diretrizes de Implementação

1. **Dimensões:** Ler os inteiros $L$ (linhas) e $C$ (colunas) do espectro de magnitude — já fornecido **centrado** (equivalente à saída de `np.fft.fftshift`).
2. **Frequência de corte:** Ler o inteiro $D_0$.
3. **Dados:** Ler os valores inteiros da matriz de magnitude, linha a linha.
4. **Centro do espectro:** Calcular $(c_y, c_x) = (L \mathbin{//} 2,\; C \mathbin{//} 2)$.
5. **Distância:** Para cada posição $(u,v)$, calcular
$$
D(u,v) = \sqrt{(u-c_y)^2 + (v-c_x)^2}
$$
6. **Máscara ideal:** Aplicar
$$
H(u,v) = \begin{cases} 1, & D(u,v) \le D_0 \\ 0, & D(u,v) > D_0 \end{cases}
$$
7. **Filtragem:** O valor de saída é $\text{mag}'(u,v) = \text{mag}(u,v) \cdot H(u,v)$.
8. **Saída:** Exibir a matriz filtrada com dimensões $L \times C$.

#### 📌 Restrições Computacionais

* **Comparação não estrita:** o critério usa $D(u,v) \le D_0$ (a fronteira pertence ao filtro, ou seja, é mantida).
* **Tipo:** todos os valores de entrada e saída são inteiros; a distância é calculada em ponto flutuante apenas internamente.
* **Sem arredondamento de magnitude:** como a entrada já é inteira e a máscara é binária (0 ou 1), a saída nunca precisa de arredondamento.

#### 🧠 Fundamentação Teórica

| Região | Distância ao centro | Efeito do filtro |
|---|---|---|
| **Centro** ($D \le D_0$) | Baixas frequências | Preservadas — estrutura global mantida |
| **Bordas** ($D > D_0$) | Altas frequências | Zeradas — textura e ruído removidos |
| **$D_0$ pequeno** | — | Imagem reconstruída ficaria muito borrada |
| **$D_0$ grande** | — | Pouca filtragem; quase toda energia preservada |

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linha 3: Inteiro $D_0$.
* Linhas seguintes: Elementos inteiros da matriz de magnitude (centrada).

**Saída:**

* Matriz filtrada em $L$ linhas e $C$ colunas, separados por espaço.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| 3<br>3<br>1<br>10 20 30<br>40 50 60<br>70 80 90 | 0 20 0<br>40 50 60<br>0 80 0 | Centro $(1,1)$. Cantos têm $D=\sqrt{2}\approx1.41 > 1$, logo são zerados; vizinhos ortogonais têm $D=1 \le 1$ e são mantidos. |
| 1<br>3<br>0<br>5 9 7 | 0 9 0 | $L=1, C=3$: centro em $(0,1)$. Apenas a própria posição central ($D=0$) sobrevive a $D_0=0$. |

In [3]:
#| label: fig-05-sim-ep0501
#| fig-cap: "Simulador EP05_01: Filtro Passa-Baixa Ideal no Espectro"
#| echo: false
#| output: true

from IPython.display import HTML
HTML("""
<div id="sim-ep0501" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0501 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0501 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0501 button:hover { background: #e8dfcf; }
  #sim-ep0501 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0501_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0501_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0501_grid_stats { display: grid; grid-template-columns: repeat(3, 1fr); gap: 10px; margin-bottom: 12px; }
  .sim-ep0501_stat_box { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 10px; padding: 10px; text-align: center; }
  .sim-ep0501_stat_label { font-size: 9.5px; color: #8a8371; text-transform: uppercase; letter-spacing: 0.04em; margin-bottom: 2px; font-weight: 700; }
  .sim-ep0501_stat_value { font-size: 16px; font-weight: 700; font-family: monospace; color: #26241d; }
  .sim-ep0501_cell { width: 42px; height: 42px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 11px; font-weight: 700; font-family: monospace; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulador EP05_01: Filtro Passa-Baixa Ideal</span>
  <span class="sim-ep0501_pill">H = (D &le; D₀) ? 1 : 0</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0501_panel" style="margin-bottom:14px;">
    
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Raio de corte (D₀): <span id="sim-ep0501_vl_d0" style="font-family:monospace; color:#26241d;">1</span>
      </label>
    </div>
    
    <input id="sim-ep0501_sl_d0" type="range" min="0" max="4" step="1" value="1">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Ajuste D₀ e observe quais posições do espectro 5&times;5 sobrevivem ao filtro.
    </div>

  </div>

  <!-- Exibição das Grades de Espectro -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap:14px; margin-bottom:14px;">
    
    <div class="sim-ep0501_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Espectro Original (Magnitude)
      </div>
      <div id="sim-ep0501_grid_orig" style="display:grid; grid-template-columns:repeat(5, 42px); gap:4px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0501_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Resultado Filtrado
      </div>
      <div id="sim-ep0501_grid_new" style="display:grid; grid-template-columns:repeat(5, 42px); gap:4px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0501_debug" class="sim-ep0501_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    –
  </div>

</div>
</div>

<script>
(function(){
  function initSim05Ep01(root){
    if (!root || root.dataset.sim05Ep01Init) return;
    root.dataset.sim05Ep01Init = "1";

    var N = 5, cy = Math.floor(N / 2), cx = Math.floor(N / 2);
    var mag = [];
    for (var i = 0; i < N; i++){
      var row = [];
      for (var j = 0; j < N; j++){
        row.push(10 * (i + 1) + j + 1);
      }
      mag.push(row);
    }

    var d0el = root.querySelector('#sim-ep0501_sl_d0');
    var d0v  = root.querySelector('#sim-ep0501_vl_d0');
    var go   = root.querySelector('#sim-ep0501_grid_orig');
    var gn   = root.querySelector('#sim-ep0501_grid_new');
    var dbg  = root.querySelector('#sim-ep0501_debug');

    function cellStyle(active){
      if (active) {
        return 'background:#eafaf1; color:#04342C; border:1px solid #a3e4d7;';
      } else {
        return 'background:#fafaf7; color:#8a8371; border:1px solid #e4dcc8;';
      }
    }

    function render(){
      var D0 = parseInt(d0el.value, 10);
      d0v.textContent = D0;
      go.innerHTML = '';
      gn.innerHTML = '';
      var kept = 0;

      for (var i = 0; i < N; i++){
        for (var j = 0; j < N; j++){
          var d = Math.sqrt((i - cy) * (i - cy) + (j - cx) * (j - cx));
          var keep = d <= D0;
          if (keep) kept++;

          var co = document.createElement('div');
          co.className = 'sim-ep0501_cell';
          co.style.cssText = cellStyle(true);
          co.textContent = mag[i][j];
          go.appendChild(co);

          var cn = document.createElement('div');
          cn.className = 'sim-ep0501_cell';
          cn.style.cssText = cellStyle(keep);
          cn.textContent = keep ? mag[i][j] : 0;
          gn.appendChild(cn);
        }
      }

      dbg.textContent = 'Centro = (' + cy + ', ' + cx + ')  |  D₀ = ' + D0 + '  |  Coeficientes mantidos: ' + kept + ' / ' + (N * N);
    }

    d0el.addEventListener('input', render);
    render();
  }

  function tryInitSim05Ep01(){
    var root = document.getElementById('sim-ep0501');
    if (root) initSim05Ep01(root); else setTimeout(tryInitSim05Ep01, 200);
  }
  tryInitSim05Ep01();
})();
</script>
""")

In [4]:
%%writefile EP05_01.py
# Código Python

Writing EP05_01.py


In [5]:
TestSuite("EP05_01.py").run()

### EP05_02 🟡 Filtro *Notch*: Removendo Picos Periódicos

Uma câmera de **inspeção industrial** captura imagens de placas de circuito, mas a fonte de alimentação da linha de produção introduz uma **interferência elétrica periódica** — um padrão de listras quase imperceptível a olho nu, mas que aparece no espectro de Fourier como **pares de picos brilhantes** simetricamente posicionados em torno do centro. A equipe de visão computacional não pode reprocessar a captura: precisa **localizar e apagar cirurgicamente** esses pares de picos no espectro, preservando todo o resto da informação útil da imagem.

Esse é o papel do **filtro rejeita-banda *notch***: diferente do passa-baixa (que afeta uma região contínua), ele ataca **pontos específicos e seus simétricos**, deixando o restante do espectro intocado.

#### 📋 Diretrizes de Implementação

1. **Dimensões:** Ler os inteiros $L$ (linhas) e $C$ (colunas) do espectro de magnitude centrado.
2. **Dados:** Ler os valores inteiros da matriz de magnitude, linha a linha.
3. **Picos:** Ler o inteiro $K$ (quantidade de pares de picos a remover).
4. **Para cada um dos $K$ picos:** ler três inteiros $\Delta v$, $\Delta u$, $r$ — deslocamento vertical, deslocamento horizontal e raio do *notch*.
5. **Centro do espectro:** $(c_y, c_x) = (L \mathbin{//} 2,\; C \mathbin{//} 2)$.
6. **Supressão simétrica:** para cada pico, zerar **todas** as posições $(u,v)$ tais que a distância ao ponto $(c_y+\Delta v,\, c_x+\Delta u)$ for $\le r$, **e também** todas as posições com distância $\le r$ ao ponto simétrico $(c_y-\Delta v,\, c_x-\Delta u)$.
7. **Saída:** Exibir a matriz resultante com dimensões $L \times C$.

#### 📌 Restrições Computacionais

* **Simetria obrigatória:** cada pico informado gera **dois** discos zerados (o ponto e seu simétrico em relação ao centro) — esquecer o simétrico é o erro mais comum.
* **Sobreposição:** se dois discos se sobrepõem, a posição permanece zerada (não há "soma" ou restauração).
* **Comparação não estrita:** uma posição é zerada se $\text{distância} \le r$.
* **Ordem de leitura:** os $K$ picos devem ser processados na ordem em que aparecem na entrada, mas o resultado final independe da ordem (operações de zerar são comutativas).

#### 🧠 Fundamentação Teórica

| Conceito | Papel no filtro *notch* |
|---|---|
| **Pico em $(\Delta v, \Delta u)$** | Frequência da interferência periódica detectada visualmente no espectro |
| **Ponto simétrico $(-\Delta v,-\Delta u)$** | Toda DFT de sinal real é hermitiana: picos sempre aparecem em pares simétricos ao centro |
| **Raio $r$** | Controla a "largura" da rejeição — $r$ grande remove mais energia ao redor do pico, mas também informação útil |

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linhas seguintes: Elementos inteiros da matriz de magnitude (centrada), $L$ linhas.
* Próxima linha: Inteiro $K$.
* $K$ linhas seguintes: três inteiros $\Delta v$, $\Delta u$, $r$ (separados por espaço).

**Saída:**

* Matriz resultante em $L$ linhas e $C$ colunas, separados por espaço.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| 5<br>5<br>1 2 3 4 5<br>6 7 8 9 10<br>11 12 13 14 15<br>16 17 18 19 20<br>21 22 23 24 25<br>1<br>1 1 0 | 1 2 3 4 5<br>6 0 8 9 10<br>11 12 13 14 15<br>16 17 18 0 20<br>21 22 23 24 25 | Centro $(c_y, c_x) = (2, 2)$. Pico informado $(\Delta v, \Delta u) = (1, 1)$ gera o ponto $(3, 3)$ (valor 19) e seu simétrico $(1, 1)$ (valor 7), ambos zerados com $r=0$ (apenas os pontos exatos). |



In [6]:
#| label: fig-05-sim-ep0502
#| fig-cap: "Simulador EP05_02: Filtro Notch"
#| echo: false
#| output: true
from IPython.display import HTML
HTML("""
<div id="sim-ep0502" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0502 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0502 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0502 button:hover { background: #e8dfcf; }
  #sim-ep0502 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0502_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0502_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0502_grid_ctrls { display: grid; grid-template-columns: repeat(auto-fit, minmax(130px, 1fr)); gap: 12px; }
  .sim-ep0502_cell { width: 42px; height: 42px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 11px; font-weight: 700; font-family: monospace; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulador EP05_02: Filtro Notch</span>
  <span class="sim-ep0502_pill">Par Simétrico</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0502_panel" style="margin-bottom:14px;">
    <div class="sim-ep0502_grid_ctrls">
      
      <div>
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
          <label style="font-size:11px; font-weight:700; color:#5e5a4a;">&Delta;v</label>
          <span id="sim-ep0502_vl_dv" style="font-family:monospace; font-weight:700; color:#26241d;">1</span>
        </div>
        <input id="sim-ep0502_sl_dv" type="range" min="-2" max="2" step="1" value="1">
      </div>

      <div>
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
          <label style="font-size:11px; font-weight:700; color:#5e5a4a;">&Delta;u</label>
          <span id="sim-ep0502_vl_du" style="font-family:monospace; font-weight:700; color:#26241d;">1</span>
        </div>
        <input id="sim-ep0502_sl_du" type="range" min="-2" max="2" step="1" value="1">
      </div>

      <div>
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
          <label style="font-size:11px; font-weight:700; color:#5e5a4a;">Raio (r)</label>
          <span id="sim-ep0502_vl_r" style="font-family:monospace; font-weight:700; color:#26241d;">0</span>
        </div>
        <input id="sim-ep0502_sl_r" type="range" min="0" max="2" step="1" value="0">
      </div>

    </div>

    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:10px; text-align:center;">
      Mova &Delta;v e &Delta;u para escolher o pico &mdash; observe que o par simétrico também é filtrado.
    </div>
  </div>

  <!-- Espectro 5x5 -->
  <div class="sim-ep0502_panel" style="text-align:center; margin-bottom:14px;">
    <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
      Espectro 5&times;5 (Vermelho = Removido pelo Filtro)
    </div>
    <div id="sim-ep0502_grid" style="display:grid; grid-template-columns:repeat(5, 42px); gap:4px; justify-content:center;"></div>
  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0502_debug" class="sim-ep0502_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim05Ep02(root){
    if (!root || root.dataset.sim05Ep02Init) return;
    root.dataset.sim05Ep02Init = "1";

    var N = 5, cy = Math.floor(N / 2), cx = Math.floor(N / 2);
    var mag = [];
    for (var i = 0; i < N; i++){
      var row = [];
      for (var j = 0; j < N; j++){
        row.push(i * 5 + j + 1);
      }
      mag.push(row);
    }

    var dv  = root.querySelector('#sim-ep0502_sl_dv');
    var du  = root.querySelector('#sim-ep0502_sl_du');
    var r   = root.querySelector('#sim-ep0502_sl_r');
    var dvv = root.querySelector('#sim-ep0502_vl_dv');
    var duv = root.querySelector('#sim-ep0502_vl_du');
    var rv  = root.querySelector('#sim-ep0502_vl_r');

    var grid = root.querySelector('#sim-ep0502_grid');
    var dbg  = root.querySelector('#sim-ep0502_debug');

    function cellStyle(kill){
      if (kill) {
        return 'background:#fdecea; color:#c0392b; border:1px solid #f5b7b1;';
      } else {
        return 'background:#eafaf1; color:#04342C; border:1px solid #a3e4d7;';
      }
    }

    function render(){
      var DV = parseInt(dv.value, 10);
      var DU = parseInt(du.value, 10);
      var R  = parseInt(r.value, 10);

      dvv.textContent = DV;
      duv.textContent = DU;
      rv.textContent  = R;

      var p1 = [cy + DV, cx + DU];
      var p2 = [cy - DV, cx - DU];

      grid.innerHTML = '';
      var removed = 0;

      for (var i = 0; i < N; i++){
        for (var j = 0; j < N; j++){
          var d1 = Math.sqrt((i - p1[0]) * (i - p1[0]) + (j - p1[1]) * (j - p1[1]));
          var d2 = Math.sqrt((i - p2[0]) * (i - p2[0]) + (j - p2[1]) * (j - p2[1]));
          var kill = (d1 <= R) || (d2 <= R);

          if (kill) removed++;

          var c = document.createElement('div');
          c.className = 'sim-ep0502_cell';
          c.style.cssText = cellStyle(kill);
          c.textContent = kill ? 0 : mag[i][j];
          grid.appendChild(c);
        }
      }

      dbg.textContent = 'Centro = (' + cy + ', ' + cx + ')  |  Pico = (' + p1[0] + ', ' + p1[1] + ')  |  Simétrico = (' + p2[0] + ', ' + p2[1] + ')  |  Removidos: ' + removed;
    }

    [dv, du, r].forEach(function(el){
      el.addEventListener('input', render);
    });

    render();
  }

  function tryInitSim05Ep02(){
    var root = document.getElementById('sim-ep0502');
    if (root) initSim05Ep02(root); else setTimeout(tryInitSim05Ep02, 200);
  }
  tryInitSim05Ep02();
})();
</script>
""")

In [7]:
%%writefile EP05_02.py
# Código Python

Writing EP05_02.py


In [8]:
TestSuite("EP05_02.py").run()

### EP05_03 🟠 Quantização DCT: a Verdadeira Fonte de Compressão

Um aplicativo de **galeria de fotos** precisa reduzir o tamanho de milhares de imagens antes de fazer *upload* para a nuvem, sem recodificar tudo do zero. O engenheiro responsável já tem os **coeficientes DCT** de cada bloco $4\times4$ calculados (a etapa cara computacionalmente já foi feita) — falta apenas aplicar a **tabela de quantização**, a etapa que realmente descarta informação e gera compressão. Coeficientes de alta frequência, menos perceptíveis ao olho humano, recebem divisores grandes e tendem a virar **zero**; coeficientes de baixa frequência, mais perceptíveis, recebem divisores pequenos e sobrevivem quase intactos.

Você vai implementar exatamente essa etapa: **quantizar e desquantizar** (dividir, arredondar, multiplicar de volta) — o coração da compressão *lossy* do JPEG.

#### 📋 Diretrizes de Implementação

1. **Dimensão do bloco:** Ler o inteiro $N$ (bloco $N \times N$).
2. **Coeficientes:** Ler a matriz $C$ de coeficientes DCT, $N$ linhas com $N$ inteiros cada (podem ser negativos).
3. **Tabela de quantização:** Ler a matriz $Q$, $N$ linhas com $N$ inteiros positivos cada.
4. **Quantização:** Para cada posição $(u,v)$, calcular o índice quantizado
$$
\tilde{C}(u,v) = \text{round}\!\left(\frac{C(u,v)}{Q(u,v)}\right)
$$
usando arredondamento padrão para o inteiro mais próximo (valores intermediários `.5` nunca ocorrem nos casos de teste).
5. **Desquantização (reconstrução):** Calcular
$$
C'(u,v) = \tilde{C}(u,v) \times Q(u,v)
$$
6. **Saída:** Exibir a matriz reconstruída $C'$, $N \times N$, inteiros.

#### 📌 Restrições Computacionais

* ***Round-trip* completo:** a saída é o coeficiente **reconstruído** ($\tilde{C} \times Q$), não o índice quantizado isolado.
* **Divisão em ponto flutuante:** a divisão $C(u,v)/Q(u,v)$ deve ser feita em ponto flutuante antes do arredondamento — divisão inteira truncada produzirá resultado incorreto.
* **Sinal preservado:** coeficientes negativos mantêm o sinal após quantização e reconstrução.
* **$Q(u,v) > 0$ sempre:** não há necessidade de tratar divisão por zero.

#### 🧠 Fundamentação Teórica

| Coeficiente | Frequência | Valor típico de $Q$ | Efeito da quantização |
|---|---|---|---|
| $C(0,0)$ | DC (média do bloco) | Pequeno | Quase sempre sobrevive — domina a energia |
| $C(u,v)$ baixo $u+v$ | Baixa frequência | Pequeno/médio | Parcialmente preservado |
| $C(u,v)$ alto $u+v$ | Alta frequência | Grande | Frequentemente vira zero — fonte da compressão |

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $N$.
* $N$ linhas seguintes: matriz $C$ (coeficientes DCT, inteiros, podem ser negativos).
* $N$ linhas seguintes: matriz $Q$ (tabela de quantização, inteiros positivos).

**Saída:**

* Matriz reconstruída $C'$, $N \times N$, inteiros separados por espaço.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| 4<br>50 10 -5 0<br>8 -3 2 1<br>0 1 0 0<br>2 0 0 -1<br>2 5 7 8<br>4 7 8 11<br>6 8 11 12<br>9 11 12 14 | 50 10 -7 0<br>8 0 0 0<br>0 0 0 0<br>0 0 0 0 | $C(0,0)=50/2=25 \to 25\times2=50$ (preservado). $C(0,2)=-5/7\approx-0.71\to-1\to-1\times7=-7$. Já $C(1,1)=-3/7\approx-0.43\to0$: zerado pela quantização — a maior parte do bloco vira zero, ilustrando a compactação de energia no canto superior esquerdo. |



In [9]:
#| label: fig-05-sim-ep0503
#| fig-cap: "Simulador EP05_03: Quantização DCT (*round-trip*)"
#| echo: false
#| output: true
from IPython.display import HTML
HTML("""
<div id="sim-ep0503" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0503 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0503 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0503 button:hover { background: #e8dfcf; }
  #sim-ep0503 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0503_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0503_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0503_cell { width: 46px; height: 36px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 11px; font-weight: 700; font-family: monospace; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulador EP05_03: Quantização DCT</span>
  <span class="sim-ep0503_pill">round(C / Q) &times; Q</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0503_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Escala de Q (Agressividade): <span id="sim-ep0503_vl_s" style="font-family:monospace; color:#26241d;">1.00&times;</span>
      </label>
    </div>
    
    <input id="sim-ep0503_sl_s" type="range" min="0.25" max="4" step="0.25" value="1">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Ajuste a escala de Q e veja quantos coeficientes sobrevivem (não-zero) após o round-trip.
    </div>
  </div>

  <!-- Exibição das Grades 4x4 -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap:14px; margin-bottom:14px;">
    
    <div class="sim-ep0503_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Coeficientes DCT (C)
      </div>
      <div id="sim-ep0503_grid_c" style="display:grid; grid-template-columns:repeat(4, 46px); gap:4px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0503_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Reconstruído (round(C / Q) &middot; Q)
      </div>
      <div id="sim-ep0503_grid_r" style="display:grid; grid-template-columns:repeat(4, 46px); gap:4px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0503_debug" class="sim-ep0503_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim05Ep03(root){
    if (!root || root.dataset.sim05Ep03Init) return;
    root.dataset.sim05Ep03Init = "1";

    var C = [[50, 10, -5, 0], [8, -3, 2, 1], [0, 1, 0, 0], [2, 0, 0, -1]];
    var Qbase = [[2, 5, 7, 8], [4, 7, 8, 11], [6, 8, 11, 12], [9, 11, 12, 14]];

    var s   = root.querySelector('#sim-ep0503_sl_s');
    var sv  = root.querySelector('#sim-ep0503_vl_s');
    var gc  = root.querySelector('#sim-ep0503_grid_c');
    var gr  = root.querySelector('#sim-ep0503_grid_r');
    var dbg = root.querySelector('#sim-ep0503_debug');

    function cell(v, faded){
      var c = document.createElement('div');
      c.className = 'sim-ep0503_cell';
      if (faded) {
        c.style.cssText = 'background:#fafaf7; color:#8a8371; border:1px solid #e4dcc8;';
      } else {
        c.style.cssText = 'background:#eafaf1; color:#04342C; border:1px solid #a3e4d7;';
      }
      c.textContent = v;
      return c;
    }

    function render(){
      var scale = parseFloat(s.value);
      sv.innerHTML = scale.toFixed(2) + '&times;';
      gc.innerHTML = '';
      gr.innerHTML = '';
      var zeros = 0, total = 16;

      for (var i = 0; i < 4; i++){
        for (var j = 0; j < 4; j++){
          gc.appendChild(cell(C[i][j], false));
          var Q = Qbase[i][j] * scale;
          var q = Math.round(C[i][j] / Q);
          var rec = Math.round(q * Q);
          if (rec === 0) zeros++;
          gr.appendChild(cell(rec, rec === 0));
        }
      }

      dbg.textContent = 'Zeros: ' + zeros + ' / ' + total + '  |  Quanto maior a escala de Q, mais zeros — maior compressão, menor qualidade.';
    }

    s.addEventListener('input', render);
    render();
  }

  function tryInitSim05Ep03(){
    var root = document.getElementById('sim-ep0503');
    if (root) initSim05Ep03(root); else setTimeout(tryInitSim05Ep03, 200);
  }
  tryInitSim05Ep03();
})();
</script>
""")

In [10]:
%%writefile EP05_03.py
# Código Python

Writing EP05_03.py


In [11]:
TestSuite("EP05_03.py").run()

### EP05_04 🔴 Implementando a DFT 2D a Partir da Definição

Um laboratório de pesquisa em **astronomia computacional** recebeu, de uma missão antiga, um pequeno sensor experimental cujos dados brutos não podem ser processados por bibliotecas modernas de FFT — o ambiente de validação é isolado e só permite operações aritméticas básicas. A equipe precisa **reimplementar a Transformada de Fourier Discreta 2D a partir da própria definição matemática**, célula por célula, para depois comparar bit a bit com `np.fft.fft2` em outro ambiente.

Este é o exercício mais conceitual da lista: não há atalhos. Você vai implementar o duplo somatório da @eq-05-dft diretamente, evidenciando *por que* a FFT existe — e o custo computacional que ela evita.

#### 📋 Diretrizes de Implementação

1. **Dimensões:** Ler os inteiros $M$ (linhas) e $N$ (colunas) da imagem $f(x,y)$.
2. **Dados:** Ler os valores inteiros de $f(x,y)$, linha a linha.
3. **DFT 2D:** Para cada par de frequências $(u,v)$ com $u=0,\ldots,M-1$ e $v=0,\ldots,N-1$, calcular
$$
F(u,v) = \sum_{x=0}^{M-1}\sum_{y=0}^{N-1} f(x,y)\, e^{-j2\pi\left(\frac{ux}{M}+\frac{vy}{N}\right)}
$$
usando a identidade de Euler $e^{-j\theta} = \cos(\theta) - j\sin(\theta)$ para separar parte real e imaginária — **não utilize nenhuma função de FFT pronta**.
4. **Magnitude:** Calcular $|F(u,v)| = \sqrt{\text{Re}(F)^2 + \text{Im}(F)^2}$ e arredondar para o inteiro mais próximo.
5. **Saída:** Exibir a matriz de magnitudes arredondadas, $M \times N$, na mesma ordem (sem `fftshift` — o DC permanece em $(0,0)$).

#### 📌 Restrições Computacionais

* **Proibido usar bibliotecas de FFT:** a implementação deve calcular os somatórios duplos explicitamente (laços aninhados), mesmo que mais lenta.
* **Sem `fftshift`:** a saída mantém a convenção crua da DFT, com o componente DC em $F(0,0)$ (canto superior esquerdo).
* **Arredondamento:** a magnitude final deve ser arredondada para o inteiro mais próximo; nos casos de teste não há ambiguidade `.5`.
* **Precisão:** pequenos erros de ponto flutuante (ordem de $10^{-6}$) antes do arredondamento são esperados e não afetam o resultado inteiro final.

#### 🧠 Fundamentação Teórica

| Elemento | Significado |
|---|---|
| $F(0,0)$ | Componente DC — soma de todos os pixels, $F(0,0) = \sum f(x,y)$ |
| Parte real $\text{Re}(F)$ | Projeção do sinal sobre cossenos |
| Parte imaginária $\text{Im}(F)$ | Projeção do sinal sobre senos |
| Complexidade desta implementação | $\mathcal{O}((MN)^2)$ — por isso a FFT, com $\mathcal{O}(MN\log(MN))$, é indispensável em imagens reais |

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $M$.
* Linha 2: Inteiro $N$.
* Linhas seguintes: Elementos inteiros de $f(x,y)$, $M$ linhas.

**Saída:**

* Matriz de magnitudes $|F(u,v)|$ arredondadas, $M \times N$, separadas por espaço.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| 2<br>2<br>1 2<br>3 4 | 10 2<br>4 0 | $F(0,0)=1+2+3+4=10$ (DC = soma total). $F(0,1)=(1-2)+(3-4)=-2 \to |F|=2$. $F(1,0)=(1+2)-(3+4)=-4\to|F|=4$. $F(1,1)=(1-2)-(3-4)=0$. |



In [12]:
#| label: fig-05-sim-ep0504
#| fig-cap: "Simulador EP05_04: DFT 2D manual"
#| echo: false
#| output: true
from IPython.display import HTML
HTML("""
<div id="sim-ep0504" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0504 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0504 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0504 button:hover { background: #e8dfcf; }
  .sim-ep0504_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0504_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0504_cell { width: 52px; height: 42px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 13px; font-weight: 700; font-family: monospace; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulador EP05_04: DFT 2D &mdash; Definição Direta</span>
  <span class="sim-ep0504_pill">&Sigma;&Sigma; f(x,y) e<sup>-j2&pi;(&hellip;)</sup></span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Instrução -->
  <div class="sim-ep0504_panel" style="margin-bottom:14px; text-align:center;">
    <div style="font-size:10.5px; color:#8a8371; font-weight:600;">
      Clique nas células de f(x,y) para alterar os valores (incrementa +1; Shift + clique decrementa -1) e veja |F(u,v)| recalculado ao vivo.
    </div>
  </div>

  <!-- Exibição das Grades 2x2 -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap:14px; margin-bottom:14px;">
    
    <div class="sim-ep0504_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        f(x,y) &mdash; Domínio Espacial
      </div>
      <div id="sim-ep0504_grid_f" style="display:grid; grid-template-columns:repeat(2, 52px); gap:6px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0504_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        |F(u,v)| &mdash; Magnitude (Sem Shift)
      </div>
      <div id="sim-ep0504_grid_F" style="display:grid; grid-template-columns:repeat(2, 52px); gap:6px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0504_debug" class="sim-ep0504_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim05Ep04(root){
    if (!root || root.dataset.sim05Ep04Init) return;
    root.dataset.sim05Ep04Init = "1";

    var f = [[1, 2], [3, 4]];
    var gf  = root.querySelector('#sim-ep0504_grid_f');
    var gF  = root.querySelector('#sim-ep0504_grid_F');
    var dbg = root.querySelector('#sim-ep0504_debug');

    function render(){
      gf.innerHTML = '';
      gF.innerHTML = '';

      for (var x = 0; x < 2; x++){
        for (var y = 0; y < 2; y++){
          (function(xx, yy){
            var c = document.createElement('div');
            c.className = 'sim-ep0504_cell';
            c.style.cssText = 'background:#eafaf1; color:#04342C; border:1px solid #a3e4d7; cursor:pointer;';
            c.textContent = f[xx][yy];
            c.addEventListener('click', function(e){
              if (e.shiftKey){ f[xx][yy]--; } else { f[xx][yy]++; }
              render();
            });
            gf.appendChild(c);
          })(x, y);
        }
      }

      var M = 2, N = 2;
      for (var u = 0; u < M; u++){
        for (var v = 0; v < N; v++){
          var re = 0, im = 0;
          for (var x = 0; x < M; x++){
            for (var y = 0; y < N; y++){
              var theta = 2 * Math.PI * (u * x / M + v * y / N);
              re += f[x][y] * Math.cos(theta);
              im -= f[x][y] * Math.sin(theta);
            }
          }
          var mag = Math.round(Math.sqrt(re * re + im * im));
          var c = document.createElement('div');
          c.className = 'sim-ep0504_cell';
          c.style.cssText = 'background:#fdecea; color:#c0392b; border:1px solid #f5b7b1;';
          c.textContent = mag;
          gF.appendChild(c);
        }
      }

      dbg.textContent = 'F(0,0) = soma de todos os pixels = ' + (f[0][0] + f[0][1] + f[1][0] + f[1][1]) + ' (componente DC)';
    }

    render();
  }

  function tryInitSim05Ep04(){
    var root = document.getElementById('sim-ep0504');
    if (root) initSim05Ep04(root); else setTimeout(tryInitSim05Ep04, 200);
  }
  tryInitSim05Ep04();
})();
</script>
""")

In [13]:
%%writefile EP05_04.py
# Código Python

Writing EP05_04.py


In [14]:
TestSuite("EP05_04.py").run()

### EP05_05 🏆 *Pipeline* JPEG Completo: DCT, Quantização e Reconstrução

Você foi contratado para criar, do zero, um **codec JPEG didático** em ambiente embarcado, sem qualquer biblioteca de imagem disponível — apenas operações matemáticas básicas. O cliente quer entender exatamente onde a qualidade é perdida e onde ela é recuperada, bloco por bloco. Este é o desafio final do capítulo: integrar **tudo** o que foi estudado — a DCT-II ortonormal, a quantização perceptual e a reconstrução via IDCT — em um único *pipeline* de ponta a ponta, processando um bloco $N \times N$ do início ao fim, exatamente como o padrão JPEG faz internamente, $8\times8$ pixels de cada vez.

#### 📋 Diretrizes de Implementação

1. **Dimensão do bloco:** Ler o inteiro $N$.
2. **Bloco original:** Ler a matriz de pixels $f(x,y)$, $N$ linhas com $N$ inteiros em $[0,255]$.
3. **Tabela de quantização:** Ler a matriz $Q$, $N \times N$ inteiros positivos.
4. **Centralização:** Subtrair 128 de cada pixel: $g(x,y) = f(x,y) - 128$.
5. **DCT-II 2D ortonormal:** Calcular
$$
C(u,v) = \alpha(u)\,\alpha(v)\sum_{x=0}^{N-1}\sum_{y=0}^{N-1} g(x,y)\,\cos\!\left[\frac{\pi(2x+1)u}{2N}\right]\cos\!\left[\frac{\pi(2y+1)v}{2N}\right]
$$
com $\alpha(0)=\sqrt{1/N}$ e $\alpha(k)=\sqrt{2/N}$ para $k>0$.
6. **Quantização:** $\tilde{C}(u,v) = \text{round}(C(u,v)/Q(u,v))$.
7. **Desquantização:** $C'(u,v) = \tilde{C}(u,v)\times Q(u,v)$.
8. **IDCT-II 2D (inversa ortonormal):** Calcular $g'(x,y)$ a partir de $C'(u,v)$ usando a transformada inversa correspondente (mesma base, somatório sobre $u,v$).
9. **Reversão da centralização e arredondamento:** $f'(x,y) = \text{round}(g'(x,y) + 128)$, restrito ao intervalo $[0,255]$ (*clipping*).
10. **Saída:** Exibir o bloco reconstruído $f'$, $N \times N$, inteiros.

#### 📌 Restrições Computacionais

* ***Pipeline* completo obrigatório:** todas as seis etapas (centralizar, DCT, quantizar, desquantizar, IDCT, reverter) devem ser implementadas — pular a quantização não passa nos testes, pois o resultado seria idêntico ao original.
* ***Clipping*:** valores reconstruídos fora de $[0,255]$ devem ser truncados (0 se negativo, 255 se maior que 255).
* **Arredondamento:** tanto na quantização quanto na reconstrução final dos pixels, use arredondamento padrão; os casos de teste evitam ambiguidade `.5`.
* **Base ortonormal:** a normalização $\alpha(u)$ e $\alpha(v)$ deve ser aplicada exatamente como especificado — sem ela, a IDCT não reconstrói corretamente.

#### 🧠 Fundamentação Teórica

| Etapa | Análoga no padrão JPEG real | Onde a qualidade é perdida |
|---|---|---|
| Centralização | Mesma — DCT assume sinal centrado em zero | Nenhuma perda |
| DCT-II | Etapa 3–4 do *pipeline* (@tbl-05-pipeline-jpeg) | Nenhuma perda (transformação exata e reversível) |
| Quantização | Etapa 5 — divisão por $Q(u,v)$ | **Principal fonte de perda** — coeficientes de alta frequência viram zero |
| IDCT | Reconstrução final | Reconstrói exatamente os coeficientes *quantizados*, não os originais |

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $N$.
* $N$ linhas seguintes: bloco original $f(x,y)$, inteiros em $[0,255]$.
* $N$ linhas seguintes: tabela de quantização $Q$, inteiros positivos.

**Saída:**

* Bloco reconstruído $f'(x,y)$, $N \times N$, inteiros em $[0,255]$, separados por espaço.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| 4<br>120 130 125 128<br>115 140 135 122<br>118 150 160 130<br>110 120 145 138<br>4 6 8 10<br>6 8 10 12<br>8 10 12 16<br>10 12 16 20 | 118 126 119 131<br>114 143 140 119<br>117 149 159 130<br>107 121 146 139 | Após DCT, quantização agressiva nas altas frequências (valores grandes de $Q$ no canto inferior direito) e reconstrução via IDCT, o bloco fica **próximo** do original, mas não idêntico — a diferença é o custo da compressão *lossy*. |

#### 💡 Dica de Depuração

Se o resultado não bater, verifique nesta ordem: (1) os coeficientes DCT brutos (antes da quantização) — eles devem reconstruir o original **exatamente** via IDCT se você pular a etapa 6–7; (2) a tabela $\alpha(u)$ — erro comum é aplicar $\sqrt{2/N}$ também para $u=0$; (3) o arredondamento da quantização, que deve ocorrer **antes** de multiplicar de volta por $Q$.


In [15]:
#| label: fig-05-sim-ep0505
#| fig-cap: "Simulador EP05_05: *Pipeline* JPEG completo em bloco"
#| echo: false
#| output: true
from IPython.display import HTML
HTML("""
<div id="sim-ep0505" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0505 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0505 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0505 button:hover { background: #e8dfcf; }
  #sim-ep0505 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0505_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0505_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0505_cell { width: 46px; height: 36px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 11px; font-weight: 700; font-family: monospace; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulador EP05_05: Pipeline JPEG (Bloco 4&times;4)</span>
  <span class="sim-ep0505_pill">DCT &rarr; Q &rarr; IDCT</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0505_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Escala de Q (1 = Tabela Base, Maior = Mais Perda): <span id="sim-ep0505_vl_s" style="font-family:monospace; color:#26241d;">1.00&times;</span>
      </label>
    </div>
    
    <input id="sim-ep0505_sl_s" type="range" min="0.5" max="5" step="0.5" value="1">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Ajuste o fator de escala de quantização e observe o bloco reconstruído se afastar (ou se aproximar) do original.
    </div>
  </div>

  <!-- Exibição das Grades 4x4 -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap:14px; margin-bottom:14px;">
    
    <div class="sim-ep0505_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Bloco Original
      </div>
      <div id="sim-ep0505_grid_o" style="display:grid; grid-template-columns:repeat(4, 46px); gap:4px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0505_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Reconstruído (DCT &rarr; Q &rarr; IDCT)
      </div>
      <div id="sim-ep0505_grid_r" style="display:grid; grid-template-columns:repeat(4, 46px); gap:4px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0505_debug" class="sim-ep0505_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim05Ep05(root){
    if (!root || root.dataset.sim05Ep05Init) return;
    root.dataset.sim05Ep05Init = "1";

    var N = 4;
    var f = [[120, 130, 125, 128], [115, 140, 135, 122], [118, 150, 160, 130], [110, 120, 145, 138]];
    var Qbase = [[4, 6, 8, 10], [6, 8, 10, 12], [8, 10, 12, 16], [10, 12, 16, 20]];

    var s   = root.querySelector('#sim-ep0505_sl_s');
    var sv  = root.querySelector('#sim-ep0505_vl_s');
    var go  = root.querySelector('#sim-ep0505_grid_o');
    var gr  = root.querySelector('#sim-ep0505_grid_r');
    var dbg = root.querySelector('#sim-ep0505_debug');

    function alpha(k){ return k === 0 ? Math.sqrt(1 / N) : Math.sqrt(2 / N); }

    function dct2(g){
      var C = [];
      for (var u = 0; u < N; u++){ C.push(new Array(N).fill(0)); }
      for (var u = 0; u < N; u++){
        for (var v = 0; v < N; v++){
          var sum = 0;
          for (var x = 0; x < N; x++){
            for (var y = 0; y < N; y++){
              sum += g[x][y] * Math.cos(Math.PI * (2 * x + 1) * u / (2 * N)) * Math.cos(Math.PI * (2 * y + 1) * v / (2 * N));
            }
          }
          C[u][v] = alpha(u) * alpha(v) * sum;
        }
      }
      return C;
    }

    function idct2(C){
      var g = [];
      for (var x = 0; x < N; x++){ g.push(new Array(N).fill(0)); }
      for (var x = 0; x < N; x++){
        for (var y = 0; y < N; y++){
          var sum = 0;
          for (var u = 0; u < N; u++){
            for (var v = 0; v < N; v++){
              sum += alpha(u) * alpha(v) * C[u][v] * Math.cos(Math.PI * (2 * x + 1) * u / (2 * N)) * Math.cos(Math.PI * (2 * y + 1) * v / (2 * N));
            }
          }
          g[x][y] = sum;
        }
      }
      return g;
    }

    function cell(v){
      var c = document.createElement('div');
      c.className = 'sim-ep0505_cell';
      c.style.cssText = 'background:#eafaf1; color:#04342C; border:1px solid #a3e4d7;';
      c.textContent = v;
      return c;
    }

    function render(){
      var scale = parseFloat(s.value);
      sv.innerHTML = scale.toFixed(2) + '&times;';
      go.innerHTML = '';
      gr.innerHTML = '';

      var g = [];
      for (var x = 0; x < N; x++){
        var row = [];
        for (var y = 0; y < N; y++){
          row.push(f[x][y] - 128);
        }
        g.push(row);
      }

      var C = dct2(g);
      var Cq = [];
      for (var u = 0; u < N; u++){
        var row = [];
        for (var v = 0; v < N; v++){
          var Qv = Qbase[u][v] * scale;
          var q = Math.round(C[u][v] / Qv);
          row.push(q * Qv);
        }
        Cq.push(row);
      }

      var gr2 = idct2(Cq);
      var diffSum = 0, n = 0;

      for (var x = 0; x < N; x++){
        for (var y = 0; y < N; y++){
          go.appendChild(cell(f[x][y]));
          var rec = Math.round(gr2[x][y] + 128);
          rec = Math.max(0, Math.min(255, rec));
          gr.appendChild(cell(rec));
          diffSum += Math.abs(rec - f[x][y]);
          n++;
        }
      }

      dbg.textContent = 'Erro médio absoluto por pixel: ' + (diffSum / n).toFixed(2) + '  |  Quanto maior a escala de Q, maior o erro de reconstrução.';
    }

    s.addEventListener('input', render);
    render();
  }

  function tryInitSim05Ep05(){
    var root = document.getElementById('sim-ep0505');
    if (root) initSim05Ep05(root); else setTimeout(tryInitSim05Ep05, 200);
  }
  tryInitSim05Ep05();
})();
</script>
""")

In [16]:
%%writefile EP05_05.py
# Código Python

Writing EP05_05.py


In [17]:
TestSuite("EP05_05.py").run()